# Advanced Feature Engineering

The first version of the NBA win probability model was able to predict outcomes using basic game-state features such as score differential, time remaining, and momentum.

While the model achieved strong baseline performance, most of its predictive power came from the current score differential. This suggests that additional basketball-specific features are needed to better represent the state of a live game.

This notebook focuses on extracting advanced features from play-by-play data to improve the model's understanding of game context.

The features introduced in this stage aim to represent:

- Which team currently has possession
- Recent scoring trends
- Late-game situations
- Strategic advantages that are not represented by the score alone

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/ml_dataset_sample.parquet")

print(df.shape)
df.head()

(2000000, 8)


,gameid,season,period,game_seconds_remaining,score_diff,score_diff_squared,momentum,home_win
0,22300188,2024,3,1255.0,0.0,0.0,-0.35,1
1,20300391,2004,1,2523.0,0.0,0.0,0.00,1
2,20301094,2004,1,2277.0,-5.0,25.0,-0.25,1
3,29600303,1997,2,1609.0,0.0,0.0,0.00,0
4,22000011,2021,4,356.0,14.0,196.0,-0.20,1


In [2]:
import pandas as pd

full_df = pd.read_parquet("../data/processed/ml_dataset.parquet")
sample_df = pd.read_parquet("../data/processed/ml_dataset_sample.parquet")

print("Full dataset:")
print(full_df.shape)
print(full_df.columns.tolist())

print("\nSample dataset:")
print(sample_df.shape)
print(sample_df.columns.tolist())

Full dataset:
(18255730, 8)
['gameid', 'season', 'period', 'game_seconds_remaining', 'score_diff', 'score_diff_squared', 'momentum', 'home_win']

Sample dataset:
(2000000, 8)
['gameid', 'season', 'period', 'game_seconds_remaining', 'score_diff', 'score_diff_squared', 'momentum', 'home_win']


# Game State Features

This section introduces features that describe the current situation of the game. These features capture high-pressure moments such as close games, late-game scenarios, and situations where a single possession can significantly impact the final outcome.

In [3]:
df["clutch"] = (
    (df["game_seconds_remaining"] <= 300) &
    (abs(df["score_diff"]) <= 5)
).astype(int)

In [4]:
df["final_two_minutes"] = (
    df["game_seconds_remaining"] <= 120
).astype(int)

In [5]:
df["one_possession_game"] = (
    abs(df["score_diff"]) <= 3
).astype(int)

In [6]:
df["is_tied"] = (
    df["score_diff"] == 0
).astype(int)

In [7]:
df["second_half"] = (
    df["period"] >= 3
).astype(int)

In [8]:
df.head()

,gameid,season,period,game_seconds_remaining,score_diff,score_diff_squared,momentum,home_win,clutch,final_two_minutes,one_possession_game,is_tied,second_half
0,22300188,2024,3,1255.0,0.0,0.0,-0.35,1,0,0,1,1,1
1,20300391,2004,1,2523.0,0.0,0.0,0.00,1,0,0,1,1,0
2,20301094,2004,1,2277.0,-5.0,25.0,-0.25,1,0,0,0,0,0
3,29600303,1997,2,1609.0,0.0,0.0,0.00,0,0,0,1,1,0
4,22000011,2021,4,356.0,14.0,196.0,-0.20,1,0,0,0,0,1


In [9]:
df.to_parquet("../data/processed/ml_dataset_features_v1.parquet")

In [10]:
new_features = [
    "clutch",
    "final_two_minutes",
    "one_possession_game",
    "is_tied",
    "second_half"
]

df[new_features].describe()

,clutch,final_two_minutes,one_possession_game,is_tied,second_half
count,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06,2.000000e+06
mean,5.858150e-02,3.588750e-02,5.870215e-01,4.383045e-01,5.076490e-01
std,2.348398e-01,1.860097e-01,4.923691e-01,4.961792e-01,4.999416e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00
75%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [11]:
df[new_features].sum()

clutch                  117163
final_two_minutes        71775
one_possession_game    1174043
is_tied                 876609
second_half            1015298
dtype: int64

In [12]:
from sklearn.model_selection import train_test_split

df_exp = df.sample(n=1_000_000, random_state=42)

features = [
    "period",
    "game_seconds_remaining",
    "score_diff",
    "score_diff_squared",
    "momentum",
    "clutch",
    "final_two_minutes",
    "one_possession_game",
    "is_tied",
    "second_half"
]

X = df_exp[features]
y = df_exp["home_win"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(800000, 10)
(200000, 10)


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

In [14]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.70471

[[48910 45730]
 [13328 92032]]

              precision    recall  f1-score   support

           0       0.79      0.52      0.62     94640
           1       0.67      0.87      0.76    105360

    accuracy                           0.70    200000
   macro avg       0.73      0.70      0.69    200000
weighted avg       0.72      0.70      0.69    200000



In [15]:
importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance

,feature,importance
2,score_diff,0.608796
4,momentum,0.151441
3,score_diff_squared,0.076845
7,one_possession_game,0.047082
0,period,0.043918
1,game_seconds_remaining,0.041882
8,is_tied,0.014566
9,second_half,0.013250
6,final_two_minutes,0.001198
5,clutch,0.001022


In [16]:
df.to_parquet(
    "../data/processed/ml_dataset_features_v1.parquet",
    index=False
)

# Recent Scoring Features

Recent scoring features measure how much each team has scored over short periods immediately preceding each game state.

Two windows will be tested:

- **2-minute scoring differential**
- **5-minute scoring differential**

These features are intended to capture short-term scoring momentum that may not be fully represented by the current score differential.

In [17]:
import os

print(os.listdir("../data/processed"))

['ml_dataset.parquet', 'ml_dataset_features_v1.parquet', 'ml_dataset_sample.parquet']


In [18]:
import os

for root, dirs, files in os.walk("../"):
    for file in files:
        if file.lower().endswith((".csv", ".parquet")):
            print(os.path.join(root, file))

../data\processed\ml_dataset.parquet
../data\processed\ml_dataset_features_v1.parquet
../data\processed\ml_dataset_sample.parquet
../data\raw\nba_pbp\pbp1997.csv
../data\raw\nba_pbp\pbp1998.csv
../data\raw\nba_pbp\pbp1999.csv
../data\raw\nba_pbp\pbp2000.csv
../data\raw\nba_pbp\pbp2001.csv
../data\raw\nba_pbp\pbp2002.csv
../data\raw\nba_pbp\pbp2003.csv
../data\raw\nba_pbp\pbp2004.csv
../data\raw\nba_pbp\pbp2005.csv
../data\raw\nba_pbp\pbp2006.csv
../data\raw\nba_pbp\pbp2007.csv
../data\raw\nba_pbp\pbp2008.csv
../data\raw\nba_pbp\pbp2009.csv
../data\raw\nba_pbp\pbp2010.csv
../data\raw\nba_pbp\pbp2011.csv
../data\raw\nba_pbp\pbp2012.csv
../data\raw\nba_pbp\pbp2013.csv
../data\raw\nba_pbp\pbp2014.csv
../data\raw\nba_pbp\pbp2015.csv
../data\raw\nba_pbp\pbp2016.csv
../data\raw\nba_pbp\pbp2017.csv
../data\raw\nba_pbp\pbp2018.csv
../data\raw\nba_pbp\pbp2019.csv
../data\raw\nba_pbp\pbp2020.csv
../data\raw\nba_pbp\pbp2021.csv
../data\raw\nba_pbp\pbp2022.csv
../data\raw\nba_pbp\pbp2023.csv
../dat

In [19]:
pbp = pd.read_csv("../data/raw/nba_pbp/pbp2025.csv")

print(pbp.shape)
print(pbp.columns.tolist())
pbp.head()

(651322, 15)
['gameid', 'period', 'clock', 'h_pts', 'a_pts', 'team', 'playerid', 'player', 'type', 'subtype', 'result', 'x', 'y', 'dist', 'desc']


,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc
0,52400111,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0,0,0,Start of 1st Period (7:44 PM EST)
1,52400111,1,PT12M00.00S,NaN,NaN,CHI,202696,N. Vučević,Jump Ball,NaN,NaN,0,0,0,Jump Ball Vucevic vs. Ware: Tip to Adebayo
2,52400111,1,PT11M45.00S,NaN,NaN,MIA,203952,A. Wiggins,Missed Shot,Driving Layup Shot,Missed,-5,29,3,MISS Wiggins 3' Driving Layup
3,52400111,1,PT11M45.00S,NaN,NaN,CHI,1630581,J. Giddey,NaN,NaN,NaN,0,0,0,Giddey BLOCK (1 BLK)
4,52400111,1,PT11M45.00S,NaN,NaN,NaN,1610612748,NaN,Rebound,Unknown,NaN,0,0,0,Heat Rebound


In [20]:
pbp["type"].value_counts()

type
Rebound           138062
Missed Shot       125371
Made Shot         109602
Substitution       64482
Free Throw         57568
Foul               50713
Turnover           37696
Timeout            14749
period             10700
Instant Replay      3165
Jump Ball           2372
Violation           2242
Ejection              94
Name: count, dtype: int64

In [21]:
pbp[pbp["type"].isin(["Made Shot", "Free Throw"])][
    ["gameid", "period", "clock", "team", "type", "subtype", "desc"]
].head(30)

,gameid,period,clock,team,type,subtype,desc
5,52400111,1,PT11M36.00S,MIA,Made Shot,Driving Finger Roll Layup Shot,Herro 2' Driving Finger Roll Layup (2 PTS)
9,52400111,1,PT11M02.00S,MIA,Made Shot,Driving Floating Jump Shot,Burks 8' Driving Floating Jump Shot (2 PTS) (A...
10,52400111,1,PT10M49.00S,CHI,Made Shot,Driving Reverse Layup Shot,White Driving Reverse Layup (2 PTS)
12,52400111,1,PT10M49.00S,CHI,Free Throw,Free Throw 1 of 1,White Free Throw 1 of 1 (3 PTS)
13,52400111,1,PT10M38.00S,MIA,Made Shot,Driving Reverse Layup Shot,Herro 2' Driving Reverse Layup (4 PTS)
16,52400111,1,PT10M23.00S,CHI,Made Shot,Putback Layup Shot,Giddey 2' Putback Layup (2 PTS)
17,52400111,1,PT10M13.00S,MIA,Made Shot,Cutting Dunk Shot,Adebayo 1' Cutting Dunk Shot (2 PTS) (Herro 1 ...
20,52400111,1,PT09M48.00S,MIA,Made Shot,Driving Layup Shot,Herro 2' Driving Layup (6 PTS) (Wiggins 1 AST)
28,52400111,1,PT09M09.00S,MIA,Made Shot,Running Jump Shot,Wiggins 25' 3PT Running Jump Shot (3 PTS) (Bur...
31,52400111,1,PT08M23.00S,MIA,Made Shot,Driving Finger Roll Layup Shot,Herro 2' Driving Finger Roll Layup (8 PTS)


In [22]:
scoring_check = pbp[pbp["type"].isin(["Made Shot", "Free Throw"])][
    ["gameid", "period", "clock", "team", "type", "h_pts", "a_pts", "desc"]
].head(30)

scoring_check

,gameid,period,clock,team,type,h_pts,a_pts,desc
5,52400111,1,PT11M36.00S,MIA,Made Shot,0.0,2.0,Herro 2' Driving Finger Roll Layup (2 PTS)
9,52400111,1,PT11M02.00S,MIA,Made Shot,0.0,4.0,Burks 8' Driving Floating Jump Shot (2 PTS) (A...
10,52400111,1,PT10M49.00S,CHI,Made Shot,2.0,4.0,White Driving Reverse Layup (2 PTS)
12,52400111,1,PT10M49.00S,CHI,Free Throw,3.0,4.0,White Free Throw 1 of 1 (3 PTS)
13,52400111,1,PT10M38.00S,MIA,Made Shot,3.0,6.0,Herro 2' Driving Reverse Layup (4 PTS)
16,52400111,1,PT10M23.00S,CHI,Made Shot,5.0,6.0,Giddey 2' Putback Layup (2 PTS)
17,52400111,1,PT10M13.00S,MIA,Made Shot,5.0,8.0,Adebayo 1' Cutting Dunk Shot (2 PTS) (Herro 1 ...
20,52400111,1,PT09M48.00S,MIA,Made Shot,5.0,10.0,Herro 2' Driving Layup (6 PTS) (Wiggins 1 AST)
28,52400111,1,PT09M09.00S,MIA,Made Shot,5.0,13.0,Wiggins 25' 3PT Running Jump Shot (3 PTS) (Bur...
31,52400111,1,PT08M23.00S,MIA,Made Shot,5.0,15.0,Herro 2' Driving Finger Roll Layup (8 PTS)


In [23]:
print(
    pbp[pbp["type"].isin(["Made Shot", "Free Throw"])][
        ["h_pts", "a_pts"]
    ].head(30).to_string(index=False)
)

 h_pts  a_pts
   0.0    2.0
   0.0    4.0
   2.0    4.0
   3.0    4.0
   3.0    6.0
   5.0    6.0
   5.0    8.0
   5.0   10.0
   5.0   13.0
   5.0   15.0
   8.0   15.0
   8.0   17.0
   9.0   17.0
  10.0   17.0
  10.0   19.0
  12.0   19.0
  12.0   22.0
  12.0   25.0
  14.0   25.0
  14.0   28.0
  16.0   28.0
  18.0   28.0
  18.0   31.0
  18.0   32.0
  18.0   33.0
  21.0   33.0
  21.0   36.0
  21.0   37.0
  21.0   38.0
  23.0   38.0


In [24]:
duplicate_times = (
    pbp.groupby(["gameid", "period", "clock"])
       .size()
       .reset_index(name="count")
)

print("Total timestamps:", len(duplicate_times))
print("Timestamps with multiple events:",
      (duplicate_times["count"] > 1).sum())

duplicate_times[duplicate_times["count"] > 1].head(20)

Total timestamps: 437101
Timestamps with multiple events: 106492


,gameid,period,clock,count
1,22400001,1,PT00M00.40S,2
3,22400001,1,PT00M26.30S,4
10,22400001,1,PT01M38.00S,2
12,22400001,1,PT01M54.00S,2
13,22400001,1,PT01M58.00S,2
14,22400001,1,PT02M19.00S,2
18,22400001,1,PT02M54.00S,4
23,22400001,1,PT03M44.00S,3
29,22400001,1,PT04M27.00S,2
30,22400001,1,PT04M39.00S,3


In [25]:
pbp[
    (pbp["gameid"] == 52400111) &
    (pbp["period"] == 1) &
    (pbp["clock"] == "PT10M49.00S")
][
    ["gameid", "period", "clock", "team", "type", "h_pts", "a_pts", "desc"]
]

,gameid,period,clock,team,type,h_pts,a_pts,desc
10,52400111,1,PT10M49.00S,CHI,Made Shot,2.0,4.0,White Driving Reverse Layup (2 PTS)
11,52400111,1,PT10M49.00S,MIA,Foul,NaN,NaN,Ware S.FOUL (P1.T1) (J.Van Duyne)
12,52400111,1,PT10M49.00S,CHI,Free Throw,3.0,4.0,White Free Throw 1 of 1 (3 PTS)


In [29]:
def clock_to_seconds(clock):
    if pd.isna(clock):
        return np.nan
    
    clock = str(clock).replace("PT", "").replace("S", "")
    
    if "M" in clock:
        minutes, seconds = clock.split("M")
        return float(minutes) * 60 + float(seconds)
    
    return float(clock)


pbp["clock_seconds"] = pbp["clock"].apply(clock_to_seconds)

# Regulation periods are 12 minutes.
# This creates seconds remaining in the game.
pbp["game_seconds_remaining"] = np.where(
    pbp["period"] <= 4,
    (4 - pbp["period"]) * 720 + pbp["clock_seconds"],
    pbp["clock_seconds"]
)

print(
    pbp[
        ["gameid", "period", "clock",
         "clock_seconds", "game_seconds_remaining"]
    ].head(20)
)

      gameid  period        clock  clock_seconds  game_seconds_remaining
0   52400111       1  PT12M00.00S          720.0                  2880.0
1   52400111       1  PT12M00.00S          720.0                  2880.0
2   52400111       1  PT11M45.00S          705.0                  2865.0
3   52400111       1  PT11M45.00S          705.0                  2865.0
4   52400111       1  PT11M45.00S          705.0                  2865.0
5   52400111       1  PT11M36.00S          696.0                  2856.0
6   52400111       1  PT11M36.00S          696.0                  2856.0
7   52400111       1  PT11M12.00S          672.0                  2832.0
8   52400111       1  PT11M08.00S          668.0                  2828.0
9   52400111       1  PT11M02.00S          662.0                  2822.0
10  52400111       1  PT10M49.00S          649.0                  2809.0
11  52400111       1  PT10M49.00S          649.0                  2809.0
12  52400111       1  PT10M49.00S          649.0   

In [30]:
pbp = pbp.sort_values(
    ["gameid", "period", "clock_seconds"],
    ascending=[True, True, False]
).copy()

# Carry the most recent known score forward
pbp["h_pts"] = (
    pbp.groupby("gameid")["h_pts"]
       .ffill()
       .fillna(0)
)

pbp["a_pts"] = (
    pbp.groupby("gameid")["a_pts"]
       .ffill()
       .fillna(0)
)

# Current score differential
pbp["score_diff"] = pbp["h_pts"] - pbp["a_pts"]

print(
    pbp[
        ["gameid", "period", "clock",
         "h_pts", "a_pts", "score_diff"]
    ].head(30)
)

        gameid  period        clock  h_pts  a_pts  score_diff
3014  22400001       1  PT12M00.00S    0.0    0.0         0.0
3015  22400001       1  PT12M00.00S    0.0    0.0         0.0
3016  22400001       1  PT11M43.00S    0.0    0.0         0.0
3017  22400001       1  PT11M43.00S    0.0    0.0         0.0
3018  22400001       1  PT11M42.00S    0.0    0.0         0.0
3019  22400001       1  PT11M38.00S    0.0    0.0         0.0
3020  22400001       1  PT11M37.00S    0.0    0.0         0.0
3021  22400001       1  PT11M24.00S    0.0    0.0         0.0
3022  22400001       1  PT11M22.00S    0.0    0.0         0.0
3023  22400001       1  PT11M17.00S    0.0    0.0         0.0
3024  22400001       1  PT10M57.00S    0.0    0.0         0.0
3025  22400001       1  PT10M57.00S    0.0    0.0         0.0
3026  22400001       1  PT10M55.00S    0.0    0.0         0.0
3027  22400001       1  PT10M55.00S    0.0    0.0         0.0
3028  22400001       1  PT10M54.00S    0.0    0.0         0.0
3029  22

In [31]:
pbp[
    (pbp["gameid"] == 52400111) &
    (pbp["period"] == 1) &
    (pbp["clock"] == "PT10M49.00S")
][
    ["clock", "team", "type",
     "h_pts", "a_pts", "score_diff"]
]

,clock,team,type,h_pts,a_pts,score_diff
10,PT10M49.00S,CHI,Made Shot,2.0,4.0,-2.0
11,PT10M49.00S,MIA,Foul,2.0,4.0,-2.0
12,PT10M49.00S,CHI,Free Throw,3.0,4.0,-1.0


In [32]:

test_game = pbp[pbp["gameid"] == 52400111].copy()

score_states = (
    test_game
    .sort_values("game_seconds_remaining", ascending=False)
    .drop_duplicates(
        subset=["period", "game_seconds_remaining"],
        keep="last"
    )
    [["gameid", "period", "game_seconds_remaining", "score_diff"]]
    .sort_values("game_seconds_remaining", ascending=False)
    .copy()
)

print(score_states.head(20))

      gameid  period  game_seconds_remaining  score_diff
1   52400111       1                  2880.0         0.0
4   52400111       1                  2865.0         0.0
6   52400111       1                  2856.0        -2.0
7   52400111       1                  2832.0        -2.0
8   52400111       1                  2828.0        -2.0
9   52400111       1                  2822.0        -4.0
12  52400111       1                  2809.0        -1.0
13  52400111       1                  2798.0        -3.0
14  52400111       1                  2787.0        -3.0
15  52400111       1                  2785.0        -3.0
16  52400111       1                  2783.0        -1.0
17  52400111       1                  2773.0        -3.0
18  52400111       1                  2761.0        -3.0
19  52400111       1                  2759.0        -3.0
21  52400111       1                  2748.0        -5.0
22  52400111       1                  2738.0        -5.0
23  52400111       1           

In [33]:

current = score_states[
    ["gameid", "game_seconds_remaining", "score_diff"]
].copy()

current["target_time"] = current["game_seconds_remaining"] + 120

history = score_states[
    ["gameid", "game_seconds_remaining", "score_diff"]
].copy()

history = history.rename(
    columns={
        "game_seconds_remaining": "past_time",
        "score_diff": "past_score_diff"
    }
)

current = current.sort_values(
    ["gameid", "target_time"]
)

history = history.sort_values(
    ["gameid", "past_time"]
)

current = pd.merge_asof(
    current,
    history,
    left_on="target_time",
    right_on="past_time",
    by="gameid",
    direction="backward"
)

current["scoring_diff_2min"] = (
    current["score_diff"] -
    current["past_score_diff"]
)

print(
    current[
        [
            "gameid",
            "game_seconds_remaining",
            "score_diff",
            "past_score_diff",
            "scoring_diff_2min"
        ]
    ].head(20)
)

      gameid  game_seconds_remaining  score_diff  past_score_diff  \
0   52400111                     0.0       -19.0            -21.0   
1   52400111                    15.5       -19.0            -21.0   
2   52400111                    17.4       -19.0            -21.0   
3   52400111                    21.1       -19.0            -21.0   
4   52400111                    27.0       -19.0            -21.0   
5   52400111                    28.5       -19.0            -21.0   
6   52400111                    49.0       -19.0            -21.0   
7   52400111                    57.9       -21.0            -21.0   
8   52400111                    75.0       -21.0            -21.0   
9   52400111                    81.0       -23.0            -21.0   
10  52400111                    83.0       -23.0            -21.0   
11  52400111                   102.0       -23.0            -23.0   
12  52400111                   107.0       -23.0            -23.0   
13  52400111                   118

## Possession Tracking (Paused)

Possession is one of the most important factors in basketball because teams cannot score without controlling the ball.

The original model does not know which team currently has possession, meaning identical game states could receive the same prediction even though one team has an immediate offensive opportunity.

This section develops a possession-tracking feature from play-by-play events and adds it as an input to the model.

In [ ]:
sample_game = df[df["gameid"] == df["gameid"].iloc[0]].copy()

sample_game = sample_game.reset_index(drop=True)

sample_game.head()

,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season
0,20000001,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0,0,0,Start of 1st Period (12:13 PM EST),2001
1,20000001,1,PT12M00.00S,0.0,0.0,NYK,948,M. Camby,Jump Ball,NaN,NaN,0,0,0,Jump Ball Camby vs. Ratliff: Tip to Houston,2001
2,20000001,1,PT11M41.00S,0.0,0.0,NYK,84,L. Sprewell,Missed Shot,Jump Shot,Missed,-58,28,6,MISS Sprewell 6' Jump Shot,2001
3,20000001,1,PT11M41.00S,0.0,0.0,PHI,689,T. Ratliff,NaN,NaN,NaN,0,0,0,Ratliff BLOCK (1 BLK),2001
4,20000001,1,PT11M40.00S,0.0,0.0,NaN,1610612755,NaN,Rebound,Unknown,NaN,0,0,0,76ers Rebound,2001


In [ ]:
teams = sample_game["team"].dropna().unique()

print(teams)

<ArrowStringArray>
['NYK', 'PHI']
Length: 2, dtype: str


In [ ]:
def other_team(current_team, teams):
    if current_team == teams[0]:
        return teams[1]
    else:
        return teams[0]

In [ ]:
sample_game["possession"] = None

In [ ]:
current_possession = None
last_shot_team = None

In [ ]:
current_possession = None

for idx, row in sample_game.iterrows():

    event = row["type"]
    team = row["team"]
    desc = str(row["desc"])


    if event == "Jump Ball":
        current_possession = team


    elif event == "Made Shot":

        last_shot_team = team

        current_possession = other_team(team, teams)


    elif event == "Missed Shot":

        last_shot_team = team


    elif event == "Turnover":
        current_possession = other_team(team, teams)


    elif event == "Rebound":

        rebound_team = None

        if pd.notna(team):
            rebound_team = team

        else:
            desc_lower = desc.lower()

            if "76ers" in desc_lower:
                rebound_team = "PHI"

            elif "knicks" in desc_lower:
                rebound_team = "NYK"

        if rebound_team is not None and last_shot_team is not None:

        
            if rebound_team != last_shot_team:
                current_possession = rebound_team

            else:
                pass


    last_shot_team = None

    sample_game.at[idx, "possession"] = current_possession

In [ ]:
sample_game[
    [
        "clock",
        "team",
        "type",
        "desc",
        "possession"
    ]
].head(40)

,clock,team,type,desc,possession
0,PT12M00.00S,NaN,period,Start of 1st Period (12:13 PM EST),None
1,PT12M00.00S,NYK,Jump Ball,Jump Ball Camby vs. Ratliff: Tip to Houston,NYK
2,PT11M41.00S,NYK,Missed Shot,MISS Sprewell 6' Jump Shot,NYK
3,PT11M41.00S,PHI,NaN,Ratliff BLOCK (1 BLK),NYK
4,PT11M40.00S,NaN,Rebound,76ers Rebound,NYK
5,PT11M29.00S,NYK,Foul,Camby S.FOUL (P1.T1),NYK
6,PT11M29.00S,PHI,Free Throw,Ratliff Free Throw 1 of 2 (1 PTS),NYK
7,PT11M29.00S,PHI,Free Throw,MISS Ratliff Free Throw 2 of 2,NYK
8,PT11M28.00S,NYK,Rebound,Ward REBOUND (Off:0 Def:1),NYK
9,PT11M18.00S,PHI,Foul,Ratliff S.FOUL (P1.T1),NYK


In [ ]:
sample_game.columns.tolist()

['gameid',
 'period',
 'clock',
 'h_pts',
 'a_pts',
 'team',
 'playerid',
 'player',
 'type',
 'subtype',
 'result',
 'x',
 'y',
 'dist',
 'desc',
 'season',
 'possession']

## Model Performance Comparison

After adding possession, the updated feature set is used to train a new Random Forest model.

Performance is compared against the Version 1 model using:

- Accuracy
- Confusion matrix
- Feature importance
- Brier score
- Probability calibration

The goal is to determine whether additional basketball context improves both prediction accuracy and probability reliability.